In [ ]:
# script for analyzing central exam outcomes for VMBO-BK and their association with exposure to school closures (determined by year of exam)

In [ ]:
import pandas as pd
import os
import numpy as np
import gc
from pandas.api.types import CategoricalDtype
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

In [ ]:
os.chdir(r'H:\Hekmat')
pd.set_option('display.max_rows', 150)

In [ ]:
df= pd.read_pickle('vmbo_bk_df.pk1')

In [ ]:
df = df[sorted(df.columns)]

In [ ]:
# examine columns
df.columns.tolist()

In [ ]:
print(df['BRINEXAMVO_crypt_2018'].unique())

In [ ]:
print(df['VOBRINVEST_2018'].unique())

In [ ]:
df['BRINEXAMVO_crypt_2018'].value_counts(dropna=False)

In [ ]:
df['VOBRINVEST_2018'].value_counts(dropna=False)

In [ ]:
df['VMBOB'].value_counts(dropna=False)

In [ ]:
df['VMBOK'].value_counts(dropna=False)

In [ ]:
# some students have multiple diploma (track) records

# identify diploma columns
dipl_cols = [col for col in df.columns if 'ONDERWIJSSOORTVODIPL_' in col]

# melt dataframe to convert from wide to long format
id_vars = ['RINPERSOON'] 
df_long_u = df.melt(id_vars=id_vars, value_vars=dipl_cols,
                  var_name='dipl_info', value_name='soort_dipl')

df_long_u = df_long_u.dropna(subset=['soort_dipl'])

# split dipl_info column into subject and year.
df_long_u[['name', 'year']] = df_long_u['dipl_info'].str.rsplit('_', n=1, expand=True)

# convert year to numeric
df_long_u['year'] = pd.to_numeric(df_long_u['year'], errors='coerce')

# get the number of repeat years
ind_counts = df_long_u['RINPERSOON'].value_counts() 
n_ind = len(ind_counts)
n_mult_records = (ind_counts > 1).sum()

mult_rec_counts = ind_counts[ind_counts >1].value_counts().sort_index()

print(f"Number of individuals: {n_ind}")
print(f"Number of individuals with multiple records: {n_mult_records}")
print("\nBreakdown of multiple records:")
print(mult_rec_counts)

In [ ]:
print(df_long_u['soort_dipl'].unique())

In [ ]:
df_long_u['soort_dipl'].value_counts(dropna=False)

In [ ]:
# identify indiv with multiple records (multiple years)
ind_counts = df_long_u['RINPERSOON'].value_counts()
mult_inds = ind_counts[ind_counts >1].index

# subset to only those indivs.
df_mult_years = df_long_u[df_long_u['RINPERSOON'].isin(mult_inds)]

# check how many unique indivs
print(f"# indivs with mult years: {len(mult_inds)}")

# preview
df_mult_years.sort_values(['RINPERSOON', 'year']).head(30)


In [ ]:
# first remove rows with other diplomas
df_long_u= df_long_u[df_long_u['soort_dipl'].isin(['Vmbo-kaderberoeps', 'Vmbo-basisberoeps'])]

In [ ]:
# identify indiv with multiple records (multiple years)
ind_counts = df_long_u['RINPERSOON'].value_counts()
mult_inds = ind_counts[ind_counts >1].index

# subset to only those indivs.
df_mult_years = df_long_u[df_long_u['RINPERSOON'].isin(mult_inds)]

# check how many unique indivs
print(f"# indivs with mult years: {len(mult_inds)}")

# preview
df_mult_years.sort_values(['RINPERSOON', 'year']).head(30)

In [ ]:
df_long_u.shape

In [ ]:
# To stay consistent with exam score selection when multiple years are present
# keep only initial designation of the diploma
df_long_u = df_long_u.sort_values(by=["RINPERSOON", "year"])
df_long_u = df_long_u.drop_duplicates(subset=["RINPERSOON"], keep="first")

In [ ]:
df_long_u.shape

In [ ]:
df.shape

In [ ]:
df_long_u.head()

In [ ]:
# merge back the list of RINPERSOONS with the initial designation of diploma along with the year

df = df.merge(df_long_u[['RINPERSOON', 'dipl_info', 'soort_dipl', 'year']], on='RINPERSOON', how='left')

In [ ]:
df.shape

In [ ]:
# check how many did get a VMBOB diploma
counts = df['VMBOB'].value_counts(dropna=False)
percentages = df['VMBOB'].value_counts(dropna=False, normalize=True) *100

result = pd.DataFrame({'count': counts, 'percent': percentages.round(2)})
print(result)

In [ ]:
# check how many did get a VMBOK diploma
counts = df['VMBOK'].value_counts(dropna=False)
percentages = df['VMBOK'].value_counts(dropna=False, normalize=True) *100

result = pd.DataFrame({'count': counts, 'percent': percentages.round(2)})
print(result)

In [ ]:
df['year'].value_counts(dropna=False)

In [ ]:
# Now that we have our target year, time to extract time-variant covariates

# define a function that extracts the values needed from the target year

def value_extract(df, column_prefix, year_col='year'):
    """
    Extracts values from wide format columns based on the target year column.

    Parameters:
    df (pd.DataFrame): the dataframe containing wide-format data.
    column_prefix (str): the prefix of the column to extract (e.g. "income", "age").
    year_col (str): the name of the column indicating the target year (default: 'year').

    Returns:
    None (modifies df in place).

    example: value_extract(df, 'SECTORVODIPL')
    """
    # ensure the year column is a string for dynamic column name lookup
    years = df[year_col].astype(str)

    # define the full range of years to check (2023 down to 2010)
    valid_years = [str(y) for y in range(2023, 2009, -1)]

    #Dynamically create column names based on available years
    column_names = [f"{column_prefix}_{y}" for y in valid_years]

    #Ensure the requested columns exist in the dataframe
    existing_columns = df.columns.intersection(column_names)

    # extract values row-wise, prioritizing specified year and falling back to previous years
    def get_value(row):
        target_col = f"{column_prefix}_{row[year_col]}"
        if target_col in existing_columns and pd.notna(row[target_col]):
            return row[target_col]
        else:
            for col in existing_columns:
                if pd.notna(row[col]):
                    return row[col]
        return np.nan # If no value is found, return NaN
    # apply function
    df[column_prefix] = df.apply(get_value, axis=1)
    

In [ ]:
# we also need to define a function that extracts the values needed from the target year only 
# ie without falling back to other years if the target year is not available. This is needed for central exam average results

def value_ex_strct(df, column_prefix, year_col='year'):
    """
    Extracts values from wide format columns based on the target year column.

    Parameters:
    df (pd.DataFrame): the dataframe containing wide-format data.
    column_prefix (str): the prefix of the column to extract (e.g. "income", "age").
    year_col (str): the name of the column indicating the target year (default: 'year').

    Returns:
    None (modifies df in place).

    example: value_extract(df, 'SECTORVODIPL')
    """
    # ensure the year column is a string for dynamic column name lookup
    years = df[year_col].astype(str)

    # define the full range of years to check (2023 down to 2010)
    valid_years = [str(y) for y in range(2023, 2009, -1)]

    #Dynamically create column names based on available years
    column_names = [f"{column_prefix}_{y}" for y in valid_years]

    #Ensure the requested columns exist in the dataframe
    existing_columns = df.columns.intersection(column_names)

    # extract values row-wise, prioritizing specified year and falling back to previous years
    def get_value_strct(row):
        target_col = f"{column_prefix}_{row[year_col]}"
        if target_col in existing_columns and pd.notna(row[target_col]):
            return row[target_col]
        return np.nan # If no value is found, return NaN
    # apply function
    df[column_prefix] = df.apply(get_value_strct, axis=1)

In [ ]:
value_extract(df, 'BRINEXAMVO_crypt')

In [ ]:
# change the odd values below to NaN
df.loc[df['OPLNIVSOI2016AGG3HBPA_2020'] == 'ÿÿÿÿÿÿïÿ', 'OPLNIVSOI2016AGG3HBPA_2020'] = np.nan
df.loc[df['OPLNIVSOI2016AGG3HBMA_2020'] == 'ÿÿÿÿÿÿïÿ', 'OPLNIVSOI2016AGG3HBMA_2020'] = np.nan

In [ ]:
value_extract(df, 'OPLNIVSOI2016AGG3HBMA')

In [ ]:
df['OPLNIVSOI2016AGG3HBMA'].value_counts(dropna=False)

In [ ]:
value_extract(df, 'OPLNIVSOI2016AGG3HBPA')

In [ ]:
df['OPLNIVSOI2016AGG3HBPA_2015'].value_counts(dropna=False)

In [ ]:
# change education into high/mid/low

low_ed = ['Vmbo-g/t, havo-, vwo-onderbouw', 'Basisonderwijs', 'Vmbo-b/k, mbo1' ]
mid_ed = ['Hbo-, wo-bachelor', 'Mbo4', 'Havo, vwo', 'Mbo2 en mbo3']
high_ed = ['Hbo-, wo-master, doctor']

In [ ]:
df["opleidpa"] = df["OPLNIVSOI2016AGG3HBPA"].map(
    lambda x: "low" if x in low_ed else
    "mid" if x in mid_ed else
    "high"if x in high_ed else
    pd.NA
)

df["opleidpa"] = pd.Categorical(df["opleidpa"], categories=["low", "mid", "high"], ordered=True)

In [ ]:
df["opleidma"] = df["OPLNIVSOI2016AGG3HBMA"].map(
    lambda x: "low" if x in low_ed else
    "mid" if x in mid_ed else
    "high"if x in high_ed else
    pd.NA
)

df["opleidma"] = pd.Categorical(df["opleidma"], categories=["low", "mid", "high"], ordered=True)

In [ ]:
df['opleidma'].value_counts(dropna=False)

In [ ]:
df['opleid_comb'] = df[['opleidpa', 'opleidma']].apply(
    lambda x: x.max(skipna=True) if not x.isna().all() else "unknown",
    axis=1
)

In [ ]:
df["opleid_comb"] = pd.Categorical(df["opleid_comb"], categories=["low", "unknown", "mid", "high"], ordered=True)

In [ ]:
df['opleid_comb'].value_counts(dropna=False)

In [ ]:
value_extract(df, 'OUDERLIJKESTRUCTUUR')

In [ ]:
df['OUDERLIJKESTRUCTUUR'].value_counts(dropna=False)

In [ ]:
value_extract(df, 'INKHP100HBESTMA')
df['INKHP100HBESTMA'].value_counts(dropna=False)

In [ ]:
value_extract(df, 'INKHP100HBESTPA')

In [ ]:
print(df['INKHP100HBESTPA'].value_counts(dropna=False))

In [ ]:
df['INKHP100HBESTPA'].dtypes

In [ ]:
df['INKHP100HBESTPA_NUM'] = pd.to_numeric(
    df['INKHP100HBESTPA'].str.extract(r'(\d+)')[0],
    errors='coerce'
).astype('float32')

In [ ]:
df['INKHP100HBESTPA_NUM'].dtypes

In [ ]:
print(df['INKHP100HBESTPA_NUM'].value_counts(dropna=False))

In [ ]:
df['INKHP100HBESTMA_NUM'] = pd.to_numeric(
    df['INKHP100HBESTMA'].str.extract(r'(\d+)')[0],
    errors='coerce'
).astype('float32')

In [ ]:
print(df['INKHP100HBESTMA_NUM'].value_counts(dropna=False))

In [ ]:
df['INKHP100HBEST_MEAN'] = df[['INKHP100HBESTMA_NUM', 'INKHP100HBESTPA_NUM']].mean(axis=1)

In [ ]:
print(df['INKHP100HBEST_MEAN'].value_counts(dropna=False))

In [ ]:
# next we convert centiles to quintiles (with NaN as a 6th category
## define function:
def perc_to_quint(p):
    if pd.isna(p):
        return 'Missing'
    elif p <= 20:
        return 'Q1'
    elif p <= 40:
        return 'Q2'
    elif p <= 60:
        return 'Q3'
    elif p <= 80:
        return 'Q4'
    else:
        return 'Q5'

quint_ord = ['Q1','Q2','Q3','Q4','Q5','Missing']

In [ ]:
# apply to mean income
df['INKHP100HBEST_QUINT'] = df['INKHP100HBEST_MEAN'].apply(perc_to_quint)
df['INKHP100HBEST_QUINT'] = pd.Categorical(df['INKHP100HBEST_QUINT'], categories=quint_ord, ordered=True)

In [ ]:
df['INKHP100HBEST_QUINT'].value_counts(dropna=False)

In [ ]:
value_extract(df, 'STEDGEM')
df['STEDGEM'].value_counts(dropna=False)

In [ ]:
df['STEDGEM'] = pd.Categorical(df['STEDGEM'],
                                    categories=["Zeer sterk (>=2500 omgevingsadressen/km2)",
                                                "Sterk (1500 tot 2500 omgevingsadressen/km2)",
                                                "Matig (1000 tot 1500 omgevingsadressen/km2)",
                                                "Weinig (500 tot 1000 omgevingsadressen/km2)",
                                                "Niet (<500 omgevingsadressen/km2)"],
                                    ordered=True)

In [ ]:
df['STEDGEM'].value_counts(dropna=False)

In [ ]:
df['STEDGEM'] = df['STEDGEM'].cat.codes.replace(-1, np.nan).astype('category')

In [ ]:
df['STEDGEM'].dtypes

In [ ]:
df['STEDGEM'].value_counts(dropna=False)

In [ ]:
df['GENERATIE_2023'] = df['GENERATIE_2023'].replace("Onbekend", np.nan)
df['GENERATIE_2023'] = pd.Categorical(df['GENERATIE_2023'])

In [ ]:
df['HERKOMST_2023'].value_counts(dropna=False)

In [ ]:
pd.crosstab(df['HERKOMST_2023'], df['GENERATIE_2023'])

In [ ]:
Herkomst_map = {
    'Marokko':'Niet-westerse',
    'Suriname':'Niet-westerse',
    'Turkije':'Niet-westerse',
    'Voormalige Nederlandse Antillen en Aruba':'Niet-westerse',
    'Overige niet-westerse landen':'Niet-westerse',
    'Overige westerse landen':'Westerse',
    'Autochtoon':'Autochtoon'
}

In [ ]:
df['HERKOMST_CAT']= df['HERKOMST_2023'].map(Herkomst_map)
df=df.copy()

In [ ]:
Generatie_map = {
    'Eerste generatie migratieachtergrond':'1st gen',
    'Tweede generatie migratieachtergrond':'2nd gen',
    'Nederlandse achtergrond':'Native'
}

In [ ]:
df['GENERATIE_2023']= df['GENERATIE_2023'].map(Generatie_map)
df=df.copy()

In [ ]:
df['GENERATIE_2023'].value_counts(dropna=False)

In [ ]:
pd.crosstab(df['HERKOMST_CAT'], df['GENERATIE_2023'])

In [ ]:
df['Migrant'] = df['GENERATIE_2023'].astype(str) + ' ' + df['HERKOMST_CAT']

In [ ]:
df['Migrant'].value_counts(dropna=False)

In [ ]:
Herkomst_cat_type = CategoricalDtype(categories=['Autochtoon','Niet-westerse','Westerse'], ordered=True)
df['HERKOMST_CAT']=df['HERKOMST_CAT'].astype(Herkomst_cat_type)

In [ ]:
Migrant_cat_type = CategoricalDtype(categories=['Native Autochtoon',
                                                '2nd gen Niet-westerse','2nd gen Westerse',
                                                '1st gen Niet-westerse','1st gen Westerse'],
                                    ordered=True)
df['Migrant']=df['Migrant'].astype(Migrant_cat_type)

In [ ]:
value_ex_strct(df, 'CEGEM')
df['CEGEM'].value_counts(dropna=False)

In [ ]:
df.shape

In [ ]:
predictors = ['GESLACHT_2023', 'Migrant', 'opleid_comb', 'STEDGEM', 'INKHP100HBEST_QUINT']
encoded = pd.get_dummies(df[predictors], drop_first=True)

corr_matrix = encoded.corr()
print(corr_matrix)

with open("vmbobk_corr_mx.txt", "w") as f:
    f.write(corr_matrix.round(2).to_string())

In [ ]:
corr_matrix.to_csv("vmbobk_corr_mx.csv", sep=";")

In [ ]:
missing_indicator=df[predictors].isna()
missing_combos = missing_indicator.groupby(predictors).size().reset_index(name='count')
print(missing_combos)

In [ ]:
# identify central exam columns
exam_cols = [col for col in df.columns if '_CE_' in col]

In [ ]:
exam_cols

In [ ]:
# melt dataframe to convert from wide to long format
id_vars = ['RINPERSOON', 'GESLACHT_2023', 'GENERATIE_2023', 'opleid_comb', 'STEDGEM', 'CEGEM', 'BRINEXAMVO_crypt', 'INKHP100HBEST_QUINT', 'HERKOMST_CAT', 'Migrant', 'year'] 

In [ ]:
df_long = df.melt(id_vars=id_vars, value_vars=exam_cols,
                  var_name='exam_info', value_name='exam_score')

In [ ]:
df_long = df_long.dropna(subset=['exam_score'])

In [ ]:
df_long.shape

In [ ]:
df_long.head()

In [ ]:
# split exam_info column into subject and year.
df_long[['subject', 'e_year']] = df_long['exam_info'].str.rsplit('_CE_', n=1, expand=True)

In [ ]:
# convert year and score to numeric
df_long['e_year'] = pd.to_numeric(df_long['e_year'], errors='coerce')
df_long['exam_score'] = pd.to_numeric(df_long['exam_score'], errors='coerce')

In [ ]:
df_long.head()

In [ ]:
sns.scatterplot(data=df_long, x='year', y='e_year')
plt.show()

In [ ]:
pd.crosstab(df_long['year'], df_long['e_year'])

In [ ]:
print(df_long.dtypes)

In [ ]:
df_long.shape

In [ ]:
df_long= df_long[df_long['year'] == df_long['e_year']]

In [ ]:
df_long.shape

In [ ]:
df_long['GENERATIE_2023'].value_counts(dropna=False)

In [ ]:
df_long['opleid_comb'].value_counts(dropna=False)

In [ ]:
# remove exam years that are before 2015

df_long = df_long[df_long['year'] >= 2015].copy()

In [ ]:
df_long.tail()

In [ ]:
# define function to assign control/exposure based on exam year

def assign_group(year):
    if 2015 <= year <= 2018:
        return 'control'
    elif year == 2019:
        return 'exposure_2019'
    elif year == 2020:
        return 'exposure_2020'
    elif year == 2021:
        return 'exposure_2021'
    elif year == 2022:
        return 'exposure_2022'
    else:
        return None

In [ ]:
# define function to assign control/exposure based on exam year

def assign_group_cat (year):
    if year == 2015:
        return 'control_2015'
    elif year == 2016:
        return 'control_2016'
    elif year == 2017:
        return 'control_2017'
    elif year == 2018:
        return 'control_2018'
    elif year == 2019:
        return 'exposure_2019'
    elif year == 2020:
        return 'exposure_2020'
    elif year == 2021:
        return 'exposure_2021'
    elif year == 2022:
        return 'exposure_2022'
    else:
        return None

In [ ]:
# define function to assign BINARY control/exposure based on exam year

def assign_group_bin (year):
    if 2015 <= year <= 2018:
        return 'control'
    elif year == 2019:
        return 'closure'
    elif year == 2020:
        return 'closure'
    elif year == 2021:
        return 'closure'
    elif year == 2022:
        return 'closure'
    else:
        return None

In [ ]:
df_long['year'].value_counts(dropna=False)

In [ ]:
df_long['cov_exp'] = df_long['year'].apply(assign_group_cat)
df_long = df_long[df_long['cov_exp'].notnull()].copy()

In [ ]:
df_long['cov_exp_bin'] = df_long['year'].apply(assign_group_bin)
df_long['cov_exp_bin'] = pd.Categorical(df_long['cov_exp_bin'], categories=['control','closure'], ordered=True)

In [ ]:
print(df_long['cov_exp'].value_counts(dropna=False))

In [ ]:
subjects = df_long['subject'].unique()
results= {}
print(subjects)

In [ ]:
df_long.shape

In [ ]:
df.shape

In [ ]:
# run GLM for mean scores
# first prep the dataframe
df_gem = df.dropna(subset=['CEGEM'], how='all').copy()

df_gem.shape

In [ ]:
df_gem['cov_exp'] = df_gem['year'].apply(assign_group_cat)

In [ ]:
df_gem['cov_exp_bin'] = df_gem['year'].apply(assign_group_bin)
df_gem['cov_exp_bin'] = pd.Categorical(df_gem['cov_exp_bin'], categories=['control','closure'], ordered=True)

In [ ]:
df_gem = df_gem[df_gem['cov_exp'].notnull()].copy()

In [ ]:
df_gem.shape

In [ ]:
print(df_gem['Migrant'].value_counts(dropna=False))

In [ ]:
print(df_gem['GESLACHT_2023'].value_counts(dropna=False))

In [ ]:
print(df_gem['opleid_comb'].value_counts(dropna=False))

In [ ]:
print(df_gem['INKHP100HBEST_QUINT'].value_counts(dropna=False))

In [ ]:
print(df_gem['STEDGEM'].value_counts(dropna=False))

In [ ]:
print(df_gem['cov_exp'].value_counts(dropna=False))

In [ ]:
# # base model:
# model = smf.glm('CEGEM ~ C(cov_exp) + GESLACHT_2023 + C(GENERATIE_2023, Treatment("Native")) + C(opleid_comb, Treatment("mid")) + STEDGEM + INKHP100HBEST_QUINT',
#                 data=df_gem).fit()
# print(model.summary())

In [ ]:
# # base model:
# model = smf.glm('CEGEM ~ C(cov_exp) + GESLACHT_2023 + C(HERKOMST_CAT) + C(opleid_comb, Treatment("mid")) + STEDGEM + INKHP100HBEST_QUINT',
#                 data=df_gem).fit()
# print(model.summary())

In [ ]:
# create z scores of the sample
mu_c = df_gem['CEGEM'].mean()
sd_c = df_gem['CEGEM'].std()
df_gem['CEGEM_cz']=(df_gem['CEGEM'] - mu_c) / sd_c

In [ ]:
# check whether CE dataset contains year 2019 (hint: it shouldn't)
subset = df_gem[df_gem['cov_exp'] == 2019]
subset_nonmissing = subset[subset['CEGEM_cz'].notna()]
print(subset_nonmissing['CEGEM_cz'].tolist())

In [ ]:
order = ['control_2015','control_2016','control_2017','control_2018',
         'exposure_2019','exposure_2020','exposure_2021', 'exposure_2022'] 

df_gem['cov_exp'] = pd.Categorical(df_gem['cov_exp'],
                                   categories=order, ordered=True)

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_gem, x='cov_exp', y='CEGEM_cz')
plt.axhline(0, color='red', linestyle='--')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
df_CE_gem = df_gem.dropna(subset=['CEGEM']).copy()

In [ ]:
df_CE_gem['cov_exp'] = df_CE_gem['cov_exp'].cat.remove_unused_categories() 

In [ ]:
# produce mean + SD of outcome variable for each SES 
ses_vars = ['cov_exp', 'cov_exp_bin', 'GESLACHT_2023','Migrant', 'opleid_comb','STEDGEM','INKHP100HBEST_QUINT']

results = {}

for var in ses_vars:
    summary = df_CE_gem.groupby(var, observed=True)['CEGEM'].agg(
        n='count',
        mean='mean',
        std='std',
    ).reset_index()
    results[var] = summary

filename = "model_results_v3.xlsx"


if not os.path.exists(filename):
    with pd.ExcelWriter(filename, engine="openpyxl", mode="w") as writer:
        for var, summary in results.items():
            sheetname = f"desc_vmbobk_{var}"[:31]
            summary.to_excel(writer, sheet_name=sheetname, index=False)
else:
    with pd.ExcelWriter(filename, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        for var, summary in results.items():
            sheetname = f"desc_vmbobk_{var}"[:31]
            summary.to_excel(writer, sheet_name=sheetname, index=False)


In [ ]:
# base model:
model = smf.glm('CEGEM_cz ~ C(cov_exp) + GESLACHT_2023 + C(Migrant, Treatment("Native Autochtoon")) + C(opleid_comb, Treatment("mid")) + C(STEDGEM) + C(INKHP100HBEST_QUINT, Treatment("Q3"))',
                data=df_CE_gem, missing='drop').fit()
print(model.summary())

In [ ]:
print(os.getcwd())

In [ ]:
print(os.listdir())

In [ ]:
with open('modsum_vmbobk.txt', 'a') as f:
    f.write("=== Model summary - base model")
    f.write(model.summary().as_text())
    f.write("\n model diagnostics")
    f.write(f"\nAIC: {model.aic}")
    f.write(f"\nBIC: {model.bic}")
    f.write("\n\n")

# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model.params.index,
    "coef": model.params.values,
    "std_err": model.bse.values,
    "z_value": model.tvalues.values,
    "P_value": model.pvalues.values,
    "ci_lower": model.conf_int()[0],
    "ci_upper": model.conf_int()[1]
}).round(4)


with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_base", index=False)

In [ ]:
# all interaction:
model = smf.glm('CEGEM_cz ~ C(cov_exp) * C(Migrant, Treatment("Native Autochtoon"))+ C(cov_exp) * GESLACHT_2023  + C(cov_exp) * C(opleid_comb, Treatment("mid")) + C(cov_exp) * STEDGEM +  C(cov_exp) * C(INKHP100HBEST_QUINT, Treatment("Q3"))',
                data=df_CE_gem).fit()
print(model.summary())

In [ ]:
with open('modsum_vmbobk.txt', 'a') as f:
    f.write("=== Model summary - all interaction model ===")
    f.write(model.summary().as_text())
    f.write("\n model diagnostics")
    f.write(f"\nAIC: {model.aic}")
    f.write(f"\nBIC: {model.bic}")
    f.write("\n\n")

# output to EXCEL

# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model.params.index,
    "coef": model.params.values,
    "std_err": model.bse.values,
    "z_value": model.tvalues.values,
    "P_value": model.pvalues.values,
    "ci_lower": model.conf_int()[0],
    "ci_upper": model.conf_int()[1]
}).round(4)


with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_int", index=False)
   

In [ ]:
# ====================================================
# MODEL DIAGNOSTICS SCRIPT - SES INTERACTIONS GLMs
#=====================================================

# ----------------------------------------------------
# 1. SETUP
# ----------------------------------------------------

formula = 'CEGEM_cz ~ C(cov_exp) * C(Migrant, Treatment("Native Autochtoon"))+ C(cov_exp) * GESLACHT_2023  + C(cov_exp) * C(opleid_comb, Treatment("mid")) + C(cov_exp) * STEDGEM +  C(cov_exp) * C(INKHP100HBEST_QUINT, Treatment("Q3"))'

# create design matrices
y, X = dmatrices(formula, data=df_CE_gem, return_type='dataframe')

# fit GLM
# model = sm.GLM(y, X, family=sm.families.Gaussian()).fit()

# ----------------------------------------------------
# 2. MULTICOLLINEARITY CHECKS
# ----------------------------------------------------

# Variance Inflation Factors
vif_data = pd.DataFrame()
vif_data["variable"]= X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Drop intercept for readability
vif_data= vif_data[vif_data["variable"] != "Intercept"]

# condition number
condition_number = np.linalg.cond(X)

# ----------------------------------------------------
# 3. MODEL FIT AND COMPARISON
# ----------------------------------------------------

# fit simpler model with migration interaction only

model_migration_int = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) * C(Migrant, Treatment("Native Autochtoon"))+ GESLACHT_2023  + C(opleid_comb, Treatment("mid")) + STEDGEM +  C(INKHP100HBEST_QUINT, Treatment("Q3"))',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()

# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_migration_int.params.index,
    "coef": model_migration_int.params.values,
    "std_err": model_migration_int.bse.values,
    "z_value": model_migration_int.tvalues.values,
    "P_value": model_migration_int.pvalues.values,
    "ci_lower": model_migration_int.conf_int()[0],
    "ci_upper": model_migration_int.conf_int()[1]
}).round(4)

with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_mig", index=False)


In [ ]:
# fit simpler model with sex interaction only

model_sex_int = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) * GESLACHT_2023 + C(Migrant, Treatment("Native Autochtoon"))  + C(opleid_comb, Treatment("mid")) + STEDGEM +  C(INKHP100HBEST_QUINT, Treatment("Q3"))',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()


# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_sex_int.params.index,
    "coef": model_sex_int.params.values,
    "std_err": model_sex_int.bse.values,
    "z_value": model_sex_int.tvalues.values,
    "P_value": model_sex_int.pvalues.values,
    "ci_lower": model_sex_int.conf_int()[0],
    "ci_upper": model_sex_int.conf_int()[1]
}).round(4)


with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_sex", index=False)

In [ ]:
# fit simpler model with parental education interaction only
model_pe_int = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) * C(opleid_comb, Treatment("mid")) + GESLACHT_2023 + C(Migrant, Treatment("Native Autochtoon")) + STEDGEM +  C(INKHP100HBEST_QUINT, Treatment("Q3"))',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()


# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_pe_int.params.index,
    "coef": model_pe_int.params.values,
    "std_err": model_pe_int.bse.values,
    "z_value": model_pe_int.tvalues.values,
    "P_value": model_pe_int.pvalues.values,
    "ci_lower": model_pe_int.conf_int()[0],
    "ci_upper": model_pe_int.conf_int()[1]
}).round(4)



with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_pe", index=False)

In [ ]:
# fit simpler model with parental income interaction only
model_pi_int = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) *  C(INKHP100HBEST_QUINT, Treatment("Q3")) + C(opleid_comb, Treatment("mid")) + GESLACHT_2023 + C(Migrant, Treatment("Native Autochtoon")) + STEDGEM',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()


# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_pi_int.params.index,
    "coef": model_pi_int.params.values,
    "std_err": model_pi_int.bse.values,
    "z_value": model_pi_int.tvalues.values,
    "P_value": model_pi_int.pvalues.values,
    "ci_lower": model_pi_int.conf_int()[0],
    "ci_upper": model_pi_int.conf_int()[1]
}).round(4)



with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_pi", index=False)

In [ ]:
# fit simpler model with urbanicity interaction only
model_urb_int = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) * STEDGEM +  C(INKHP100HBEST_QUINT, Treatment("Q3")) + C(opleid_comb, Treatment("mid")) + GESLACHT_2023 + C(Migrant, Treatment("Native Autochtoon"))',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()



# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_urb_int.params.index,
    "coef": model_urb_int.params.values,
    "std_err": model_urb_int.bse.values,
    "z_value": model_urb_int.tvalues.values,
    "P_value": model_urb_int.pvalues.values,
    "ci_lower": model_urb_int.conf_int()[0],
    "ci_upper": model_urb_int.conf_int()[1]
}).round(4)



with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_urb", index=False)

In [ ]:
# fit basic model (no interactions)
model_basic = sm.GLM.from_formula(
    'CEGEM_cz ~ C(cov_exp) + C(Migrant, Treatment("Native Autochtoon"))+ GESLACHT_2023  + C(opleid_comb, Treatment("mid")) + STEDGEM +  C(INKHP100HBEST_QUINT, Treatment("Q3"))',
    data=df_CE_gem, family=sm.families.Gaussian()
).fit()


# extract coefficients and related statistics
results_df = pd.DataFrame({
    "term": model_basic.params.index,
    "coef": model_basic.params.values,
    "std_err": model_basic.bse.values,
    "z_value": model_basic.tvalues.values,
    "P_value": model_basic.pvalues.values,
    "ci_lower": model_basic.conf_int()[0],
    "ci_upper": model_basic.conf_int()[1]
}).round(4)



with pd.ExcelWriter(
    "model_results_v3.xlsx",
    engine="openpyxl",
    mode='a', # append to existing file
    if_sheet_exists="replace"
) as writer:
    results_df.to_excel(writer, sheet_name="Vmbobk_base", index=False)

In [ ]:
# create r2

try:
    r2model = model.rsquared
except AttributeError:
    r2model= 1 - (model.deviance / model.null_deviance)

try:
    r2mig = model_migration_int.rsquared
except AttributeError:
    r2mig= 1 - (model_migration_int.deviance / model_migration_int.null_deviance)

try:
    r2sex = model_sex_int.rsquared
except AttributeError:
    r2sex= 1 - (model_sex_int.deviance / model_sex_int.null_deviance)

try:
    r2pe = model_pe_int.rsquared
except AttributeError:
    r2pe= 1 - (model_pe_int.deviance / model_pe_int.null_deviance)

try:
    r2pi = model_pi_int.rsquared
except AttributeError:
    r2pi= 1 - (model_pi_int.deviance / model_pi_int.null_deviance)

try:
    r2urb = model_urb_int.rsquared
except AttributeError:
    r2urb= 1 - (model_urb_int.deviance / model_urb_int.null_deviance)
    
try:
    r2basic = model_basic.rsquared
except AttributeError:
    r2basic= 1 - (model_basic.deviance / model_basic.null_deviance)

comparison = pd.DataFrame({
    "Model": ["Multi-Interaction", "Single-Interaction(Migration)", "Single-Interaction(Sex)", "Single-Interaction(ParentalEdu)", "Single-Interaction(ParentalInc)", "Single-Interaction(Urbanicity)", "Base model"],
    "N": [model.nobs, model_migration_int.nobs, model_sex_int.nobs, model_pe_int.nobs, model_pi_int.nobs, model_urb_int.nobs, model_basic.nobs],
    "AIC": [model.aic, model_migration_int.aic, model_sex_int.aic, model_pe_int.aic, model_pi_int.aic, model_urb_int.aic,  model_basic.aic],
    "BIC": [model.bic, model_migration_int.bic, model_sex_int.bic, model_pe_int.bic, model_pi_int.bic, model_urb_int.bic,  model_basic.bic],
    "R2": [r2model, r2mig, r2sex, r2pe, r2pi, r2urb, r2basic],
    "Condition_Num": [np.linalg.cond(model.model.exog), np.linalg.cond(model_migration_int.model.exog), np.linalg.cond(model_sex_int.model.exog), np.linalg.cond(model_pe_int.model.exog), np.linalg.cond(model_pi_int.model.exog), np.linalg.cond(model_urb_int.model.exog), np.linalg.cond(model_basic.model.exog)]
})

# ----------------------------------------------------
# 4. SAVE DIAGNOSTIC OUTPUT
# ----------------------------------------------------

with pd.ExcelWriter("diagnostic_report.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vif_data.to_excel(writer, sheet_name="VIF_vmbobk", index=False)
    corr_matrix.to_excel(writer, sheet_name="CorrelationMatrix_vmbobk")
    comparison.to_excel(writer, sheet_name="ModelComparison_vmbobk", index=False)

    # having to expand columns everytime you examine a new sheet is annoying: access the workbook and adjust column widths
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]

        for col in worksheet.columns:
            max_length = 0
            column = col[0].column_letter # get the column name
            for cell in col:
                try:
                    # measure cell value length
                    cell_length = len(str(cell.value))
                    if cell_length > max_length:
                        max_length = cell_length
                except:
                    pass
            padded_width = max_length + 2 #add some padding
            worksheet.column_dimensions[column].width = padded_width
                        

# ----------------------------------------------------
# 5. PRINT SUMMARY
# ----------------------------------------------------

print("\n=== MODEL SUMMARY ===")
print(model.summary().tables[0])
print("\n=== CONDITION NUMBER ===")
print(round(condition_number, 2))
print("\n=== HIGH VIFs (>5) ===")
print(vif_data[vif_data["VIF"] > 5].sort_values("VIF", ascending=False))
print("\n=== MODEL SUMMARY ===")
print(comparison)

In [ ]:
## Multiple regression per CE subject

# prepare storage containers
desc_rows = []
counts_rows = []
model_rows = []

for subj in subjects:
    df_sub = df_long[df_long['subject'] == subj].copy()

    if df_sub.empty:
        print(f" skipping {subj}: no rows found.")
        continue

    # calculate control group mean and std
    mu_c = df_sub['exam_score'].mean()
    sd_c = df_sub['exam_score'].std()

    # standardize exam scores relative to contol group
    df_sub['exam_score_cz'] = (df_sub['exam_score'] - mu_c) / sd_c

    # -------- descriptive statistics ------

    desc_rows.append({
        'subject': subj,
        'mean_exam': mu_c,
        'std_exam': sd_c,
        'n': len(df_sub)
    })

    # ------ value counts for categorical predictors ------
    categorical_vars = [
        'cov_exp', 'GESLACHT_2023', 'Migrant', 'opleid_comb', 'STEDGEM', 'INKHP100HBEST_QUINT'
    ]

    for var in categorical_vars:
        counts = df_sub[var].value_counts(dropna=False)
        for cat, cnt in counts.items():
            counts_rows.append({
                'subject': subj,
                'variable': var,
                'category': cat,
                'count': cnt
            })

    # ---- Model ----
    try:
        model = smf.glm(
            'exam_score_cz ~  C(cov_exp) * C(Migrant, Treatment("Native Autochtoon"))+ C(cov_exp) * GESLACHT_2023  + C(cov_exp) * C(opleid_comb, Treatment("mid")) + C(cov_exp) * STEDGEM +  C(cov_exp) * C(INKHP100HBEST_QUINT, Treatment("Q3"))',
            data=df_sub
        ).fit()
    except Exception as e:
        print("skipping {subj}: model fitting failed ({e})")
        continue

    # store AIC/BIC
    model_rows.append({'subject': subj, 'term': 'Model', 'coef': None,
                       'std_err': None, 'z': None, 'p_value': None,
                       'CI_low': None, 'CI_high': None,
                       'AIC': model.aic, 'BIC': model.bic})
    
    # store coefficients and p-values
    coef_table = model.summary2().tables[1]
    ci = model.conf_int()
    coef_table['CI_low'] = ci[0]
    coef_table['CI_high'] = ci[1]

    for param, row in coef_table.iterrows():
        model_rows.append({
            'subject': subj,
            'term': param,
            'coef': row['Coef.'],
            'std_err': row['Std.Err.'],
            'z': row['z'],
            'p_value': row['P>|z|'],
            'CI_low': row['CI_low'],
            'CI_high': row['CI_high'],
            'AIC': None,
            'BIC': None
        })

In [ ]:
# ---- Export all results to excel ----
output_file = "subject_results_vmbobk.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # convert lists to dataframes
    df_desc=pd.DataFrame(desc_rows)
    df_counts=pd.DataFrame(counts_rows)
    df_model=pd.DataFrame(model_rows)

    #write each to a sheet
    df_desc.to_excel(writer, sheet_name='Descriptives', index=False)
    df_counts.to_excel(writer, sheet_name='Value_counts', index=False)
    df_model.to_excel(writer, sheet_name='Model_coeffs', index=False)

    #Estimate column widths for each sheet
    from openpyxl.utils import get_column_letter
    for sheet_name, df in {
        'Descriptives': df_desc,
        'Value_counts': df_counts,
        'Model_coeffs': df_model
    }.items():
        worksheet = writer.sheets[sheet_name]
        for i, col in enumerate(df.columns, start=1):
            try:
                max_len = max(df[col].astype(str).map(len).max(), len(col))+2
                worksheet.column_dimensions[get_column_letter(i)].width = min(max_len, 50)
            except Exception:
                worksheet.column_dimensions[get_column_letter(i)].width = 15 #fallback width

print(f"Results sucessfully written to {output_file}")
    